In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Fáze Transpileru

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Tato stránka popisuje fáze předpřipraveného transpilačního pipeline v Qiskit SDK. Existuje šest fází:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Funkce [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) vytvoří přednastavený [staged pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) složený z těchto fází. Konkrétní průchody tvořící každou fázi závisí na argumentech předaných funkci `generate_preset_pass_manager`. Argument `optimization_level` je poziční a musí být zadán; jde o celé číslo, které může být 0, 1, 2 nebo 3. Vyšší hodnoty znamenají intenzivnější, ale nákladnější optimalizaci (viz [Výchozí nastavení Transpileru a možnosti konfigurace](defaults-and-configuration-options)).

Doporučený způsob transpilace obvodu je vytvoření předpřipraveného staged pass manageru a jeho následné spuštění na obvodu, jak je popsáno v části [Transpilace s pass managery](transpile-with-pass-managers). Jednodušší, ale méně přizpůsobitelnou alternativou je použití funkce [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Tato funkce přijímá obvod přímo jako argument. Stejně jako u `generate_preset_pass_manager` závisí použité průchody Transpileru na argumentech, jako je `optimization_level`, předaných funkci `transpile`. Interně funkce `transpile` volá `generate_preset_pass_manager`, aby vytvořila předpřipravený staged pass manager, a spustí ho na obvodu.
## Fáze Init
Tato první fáze toho ve výchozím nastavení příliš nedělá a je primárně užitečná, pokud chceš zahrnout vlastní počáteční optimalizace. Protože většina algoritmů pro layout a routing je navržena tak, aby pracovala pouze s jednoQubitovými a dvouQubitovými Gate, tato fáze se také používá k překladu Gate operujících na více než dvou Qubitech na Gate operující pouze na jednom nebo dvou Qubitech.

Další informace o implementaci vlastních počátečních optimalizací pro tuto fázi najdeš v části o pluginech a přizpůsobení pass managerů.
## Fáze Layout
Další fáze se týká layoutu nebo konektivity Backend, na který bude Circuit odeslán. Obecně jsou kvantové Circuit abstraktní entity, jejichž Qubity jsou „virtuální" nebo „logické" reprezentace skutečných Qubitů používaných při výpočtech. Pro provedení posloupnosti Gate je nutné mapování 1:1 z „virtuálních" Qubitů na „fyzické" Qubity ve skutečném kvantovém zařízení. Toto mapování je uloženo jako objekt `Layout` a je součástí omezení definovaných v [instrukční sadě architektury (ISA)](./transpile#instruction-set-architecture) Backend.

![Tento obrázek ilustruje mapování Qubitů z reprezentace drátů do diagramu, který znázorňuje způsob propojení Qubitů na QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Mapování Qubitů")

Volba mapování je nesmírně důležitá pro minimalizaci počtu operací SWAP potřebných k mapování vstupního Circuit na topologii zařízení a pro zajištění použití nejlépe kalibrovaných Qubitů. Vzhledem k důležitosti této fáze přednastavené pass managery zkouší několik různých metod, jak najít nejlepší layout. Obvykle to zahrnuje dva kroky: nejprve se pokusí najít „perfektní" layout (layout, který nevyžaduje žádné operace SWAP), a poté heuristický průchod, který se pokusí najít nejlepší layout, pokud perfektní layout nelze nalézt. Pro tento první krok se typicky používají dva průchody `Passes`:

- `TrivialLayout`: Naivně mapuje každý virtuální Qubit na fyzický Qubit se stejným číslem na zařízení (tj. [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). Jde o historické chování používané pouze při `optimzation_level=1` k pokusu o nalezení perfektního layoutu. Pokud se to nepodaří, zkusí se jako další `VF2Layout`.
- `VF2Layout`: Jde o `AnalysisPass`, který vybírá ideální layout tím, že tuto fázi chápe jako problém izomorfismu podgrafů řešeného algoritmem VF2++. Pokud je nalezeno více než jedno mapování, spustí se hodnoticí heuristika, která vybere mapování s nejnižší průměrnou chybovostí.

Poté se pro heuristickou fázi ve výchozím nastavení používají dva průchody:

- `DenseLayout`: Najde podgraf zařízení s největší konektivitou, který má stejný počet Qubitů jako Circuit (používá se pro optimalizační úroveň 1, pokud jsou v Circuit přítomny operace řídicího toku, jako je `IfElseOp`).
- `SabreLayout`: Tento průchod vybírá layout tak, že začíná od počátečního náhodného layoutu a opakovaně spouští algoritmus `SabreSwap`. Tento průchod se používá pouze na optimalizačních úrovních 1, 2 a 3, pokud není pomocí průchodu `VF2Layout` nalezen perfektní layout. Další podrobnosti o tomto algoritmu najdeš v článku [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)
## Fáze Routing
Aby bylo možné provést dvouQubitovou Gate mezi Qubity, které nejsou přímo propojeny na kvantovém zařízení, je nutné do Circuit vložit jednu nebo více Gate SWAP, které přesunou stavy Qubitů tak, aby byly sousední na mapě Gate zařízení. Každá Gate SWAP představuje nákladnou a hlučnou operaci k provedení. Nalezení minimálního počtu Gate SWAP potřebných k mapování Circuit na dané zařízení je tedy důležitým krokem v procesu transpilace. Z důvodu efektivity se tato fáze ve výchozím nastavení typicky počítá společně s fází Layout, ale jsou od sebe logicky odděleny. Fáze *Layout* vybírá fyzické Qubity, které se mají použít, zatímco fáze *Routing* vkládá odpovídající počet Gate SWAP, aby bylo možné Circuit provést s použitím vybraného layoutu.

Nalezení optimálního mapování SWAP je však obtížné. Ve skutečnosti jde o NP-těžký problém, a proto je jeho výpočet pro všechna kromě nejmenších kvantových zařízení a vstupních Circuit prohibitivně nákladný. Aby Qiskit toto obešel, používá stochastický heuristický algoritmus `SabreSwap` pro výpočet dobrého, avšak ne nutně optimálního, mapování SWAP. Použití stochastické metody znamená, že generované Circuit nejsou zaručeně stejné při opakovaném spuštění. Opakované spuštění stejného Circuit skutečně vede k distribuci hloubek Circuit a počtů Gate na výstupu. Právě z tohoto důvodu mnoho uživatelů volí spuštění funkce routing (nebo celého `StagedPassManager`) mnohokrát a výběr Circuit s nejmenší hloubkou z distribuce výstupů.

Vezměme si například 15-Qubitový GHZ Circuit spuštěný 100krát s „špatným" (odpojeným) `initial_layout`.

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Jak vidíš, tento Circuit musí provést dvouQubitovou Gate mezi Qubity 0 a 14, které jsou na grafu konektivity velmi daleko od sebe. Spuštění tohoto Circuit tedy vyžaduje vložení Gate SWAP k provedení všech dvouQubitových Gate pomocí průchodu `SabreSwap`.

Všimni si také, že algoritmus `SabreSwap` se liší od větší metody `SabreLayout` z předchozí fáze. Ve výchozím nastavení `SabreLayout` spouští jak layout, tak routing a vrací transformovaný Circuit. To se dělá z několika konkrétních technických důvodů uvedených na [referenční stránce API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) průchodu.
## Fáze překladu
Při psaní kvantového obvodu můžeš volně používat libovolnou kvantovou bránu (unitární operaci), spolu s kolekcí nebránových operací, jako jsou instrukce pro měření qubitů nebo jejich resetování. Většina kvantových zařízení však nativně podporuje jen hrstku kvantových bránových a nebránových operací. Tyto nativní brány jsou součástí definice [ISA](./transpile#instruction-set-architecture) cíle a tato fáze přednastavených `PassManagerů` překládá (neboli *rozbaluje*) brány zadané v obvodu do nativní sady bází bran příslušného Backend. Jde o důležitý krok, protože umožňuje Backend obvod spustit, ale zpravidla vede ke zvýšení hloubky a počtu bran.

Zvláště důležité jsou dva speciální případy, které dobře ilustrují, co tato fáze dělá.

1. Pokud brána SWAP není nativní branou cílového Backend, jsou k její realizaci potřeba tři brány CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

Jako součin tří bran CNOT je SWAP nákladnou operací na šumových kvantových zařízeních. Takovéto operace jsou však obvykle nezbytné pro vložení obvodu do omezené bránové konektivity mnoha zařízení. Minimalizace počtu bran SWAP v obvodu je proto primárním cílem transpilačního procesu.

2. Toffoliho brána neboli brána řízená-řízená-not (`ccx`) je tříqubitová brána. Protože naše sada bází obsahuje pouze jedno- a dvouqubitové brány, musí být tato operace rozložena. Je to však poměrně nákladné:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Za každou Toffoliho bránu v kvantovém obvodu může hardware provést až šest bran CNOT a několik jednoqubitových bran. Tento příklad ukazuje, že každý algoritmus využívající více Toffoliho bran skončí jako obvod s velkou hloubkou, a tedy bude výrazně ovlivněn šumem.

## Fáze optimalizace
Tato fáze se soustředí na rozklad kvantových obvodů do sady bází bran cílového zařízení a musí čelit zvýšené hloubce způsobené fázemi layoutu a routování. Naštěstí existuje mnoho rutin pro optimalizaci obvodů kombinováním nebo eliminováním bran. V některých případech jsou tyto metody tak účinné, že výstupní obvody mají menší hloubku než vstupní, a to i po mapování layoutu a routování na hardwarovou topologii. V jiných případech toho moc udělat nelze a výpočet může být na šumových zařízeních obtížně proveditelný. Právě v této fázi se začínají lišit jednotlivé úrovně optimalizace.

- Pro `optimization_level=1` tato fáze připraví [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) a [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), které kombinují řetězce jednoqubitových bran a ruší sousední brány CNOT.
- Pro `optimization_level=2` tato fáze používá místo `CXCancellation` průchod [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation), který odstraňuje redundantní brány využíváním komutačních vztahů.
- Pro `optimization_level=3` tato fáze připraví následující průchody:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

Kromě toho tato fáze také provádí několik závěrečných kontrol, aby se ujistila, že všechny instrukce v obvodu jsou složeny z bází bran dostupných na cílovém Backend.

Níže uvedený příklad se stavem GHZ demonstruje vliv různých nastavení úrovně optimalizace na hloubku obvodu a počet bran.

> **Note:** Výstup transpilace se liší kvůli stochastickému SWAP mapperu. Čísla níže se proto pravděpodobně změní při každém spuštění kódu.

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubitový stav GHZ před transpilací")

Následující kód sestaví 15-qubitový stav GHZ a porovnává `optimization_levels` transpilace z hlediska výsledné hloubky obvodu, počtu bran a počtu vícequbitových bran.